# Train the "Hey Evie / Hey EV / Evie / EV" wake word (Colab or Kaggle)

One small openWakeWord model that answers to all four phrasings, trained on Piper-synthesised speech (EV's own
`hfc_female` voice gets the biggest share) plus real LibriSpeech speech and ESC-50 noise as negatives.
Output: **`hey_evie.onnx`** (~400 KB). Drop it in `models/openwakeword/` and set `trigger.wakeword_model = "hey_evie"`.

**Setup**
* **Colab**: Runtime -> CPU is fine (no GPU needed). Run the cells top to bottom; the first one asks you to upload `ev_wakeword_kit.zip`.
* **Kaggle**: Settings -> **Internet: On**. Add `ev_wakeword_kit.zip` as a dataset (Add Data -> Upload; Kaggle unzips it, that's fine). Accelerator can stay *None*.

Takes roughly 1-2 h on CPU (synthesis and embedding dominate; training itself is a minute or two).

In [ ]:
import glob, os, shutil, sys, zipfile

IN_KAGGLE = os.path.exists("/kaggle/working")
ROOT = "/kaggle/working/ev" if IN_KAGGLE else "/content/ev"
os.makedirs(ROOT, exist_ok=True)

def find_kit():
    # Kaggle unzips uploaded datasets into a folder, Colab keeps the zip
    folder = glob.glob("/kaggle/input/**/scripts/wakeword/synth.py", recursive=True)
    if folder:
        shutil.copytree(folder[0].rsplit("/scripts/", 1)[0], ROOT, dirs_exist_ok=True)
        return True
    for z in glob.glob("/kaggle/input/**/ev_wakeword_kit.zip", recursive=True) + glob.glob("/content/ev_wakeword_kit.zip"):
        zipfile.ZipFile(z).extractall(ROOT)
        return True
    return False

if not find_kit() and not IN_KAGGLE:
    from google.colab import files
    files.upload()                      # pick ev_wakeword_kit.zip
    find_kit()
assert os.path.exists(f"{ROOT}/scripts/wakeword/synth.py"), "kit not found: upload ev_wakeword_kit.zip first"
os.chdir(ROOT)
print("working in", os.getcwd(), "| cpus:", os.cpu_count())

In [ ]:
# Same versions as the local EV install. openwakeword goes in with --no-deps so it doesn't try to pull tflite on Linux.
!pip install -q piper-tts==1.8.0 kokoro-onnx==0.6.1 soundfile scipy onnx onnxruntime scikit-learn requests tqdm
!pip install -q --no-deps openwakeword==0.6.0
import torch; print("torch", torch.__version__)

In [ ]:
# Models: 6 Piper voices (hfc_female = EV's main voice), openWakeWord's shared feature models, Kokoro (evaluation only).
import subprocess
voices = ["en_US-hfc_female-medium", "en_US-amy-medium", "en_US-lessac-medium", "en_US-ryan-medium",
          "en_GB-alan-medium", "en_US-libritts_r-medium"]
for i, v in enumerate(voices):
    cmd = [sys.executable, "scripts/download_models.py", "--voice", v] + (["--wakeword", "hey_jarvis"] if i == 0 else [])
    subprocess.run(cmd, check=True)
os.makedirs("models/kokoro", exist_ok=True)
K = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0"
!wget -q -nc -P models/kokoro {K}/kokoro-v1.0.onnx {K}/voices-v1.0.bin
!ls -la models/piper models/openwakeword models/kokoro | grep -v json

In [ ]:
# Public negatives: LibriSpeech dev-clean (train) + test-clean (held out) + ESC-50 noise. ~1.3 GB, a few minutes.
!python scripts/wakeword/fetch_data.py

In [ ]:
# Piper synthesis: positives, near-miss words ("every", "Eddie", "TV", "hey Siri"...), generic speech. Resumable.
# Use fewer clips (e.g. --pos 4000 --adv 3000 --gen 3000) for a faster first pass.
!python scripts/wakeword/synth.py --pos 8000 --adv 6000 --gen 6000

In [ ]:
# Augment (noise, reverb, EQ, pitch/speed) and compute the 16x96 openWakeWord embeddings.
# If data/wakeword/recordings/{pos,neg} was in the kit, your own voice is mixed in here (very worthwhile).
!python scripts/wakeword/features.py --reps 3

In [ ]:
!python scripts/wakeword/train.py --epochs 60

In [ ]:
# Streamed through openwakeword.Model like EV does: recall on UNSEEN Kokoro voices, false triggers/hour on
# LibriSpeech test-clean + unseen noise. Pick a threshold from this table for trigger.wakeword_threshold.
!python scripts/wakeword/evaluate.py --hours 2

In [ ]:
# Grab the model. Colab downloads it; on Kaggle it appears under the notebook's Output tab.
src = "models/openwakeword/hey_evie.onnx"
if IN_KAGGLE:
    shutil.copy(src, "/kaggle/working/hey_evie.onnx")
    print("saved /kaggle/working/hey_evie.onnx (Output tab, after 'Save Version')")
else:
    from google.colab import files
    files.download(src)

**Next (on your PC):** copy `hey_evie.onnx` to `models/openwakeword/`, set in `config/ev.toml`
`wakeword_model = "hey_evie"` and a `wakeword_threshold` from the table above, then `python -m ev --debug`.